# Wikipedia Dump → JSONL Pipeline

**Repo:** [ai-wiki-dataset-preprocessor](https://github.com/TylrDn/ai-wiki-dataset-preprocessor)  
**Phase:** 3 — Notebook Pipelines  
**Output:** Sharded JSONL files + HuggingFace Dataset export

## Pipeline

```
Wikipedia XML dump (.xml.bz2)
    → wikiextractor (HTML strip)
    → clean / normalise text
    → deduplicate (MinHash)
    → shard to JSONL
    → push to HuggingFace Hub
```

### References
- HuggingFace Datasets: Wolf et al. (2020) https://arxiv.org/abs/1910.03771
- Wikimedia dumps: https://dumps.wikimedia.org/

In [ ]:
from __future__ import annotations

import logging
import os
from pathlib import Path

import wandb

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

# --- Config -----------------------------------------------------------
DUMP_PATH = Path(os.getenv("WIKI_DUMP_PATH", "data/raw/enwiki-latest-pages-articles.xml.bz2"))
OUTPUT_DIR = Path(os.getenv("WIKI_OUTPUT_DIR", "data/processed/wiki_jsonl"))
HF_REPO_ID = os.getenv("HF_REPO_ID", "TylrDn/wiki-preprocessed")
SHARD_SIZE = int(os.getenv("WIKI_SHARD_SIZE", "10000"))  # articles per shard
WANDB_DISABLED = os.getenv("WANDB_DISABLED", "false").lower() == "true"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
logger.info("Output directory: %s", OUTPUT_DIR)

In [ ]:
# Initialise Weights & Biases
# Reference: https://wandb.ai
if not WANDB_DISABLED:
    run = wandb.init(
        project="ai-wiki-dataset-preprocessor",
        config={
            "dump_path": str(DUMP_PATH),
            "output_dir": str(OUTPUT_DIR),
            "shard_size": SHARD_SIZE,
        },
    )
    logger.info("W&B run: %s", run.url)
else:
    logger.info("W&B disabled — running offline")

In [ ]:
# Install dependencies (run once)
# !pip install wikiextractor datasets huggingface_hub datasketch

In [ ]:

import bz2
import hashlib
import json
import re
import unicodedata
from typing import Generator


def iter_wiki_articles(dump_path: Path) -> Generator[dict, None, None]:
    """Stream articles from a Wikipedia XML BZ2 dump.

    Yields dicts with keys: id (str), title (str), text (str).
    Relies on wikiextractor for HTML/markup stripping.
    """
    # wikiextractor writes one JSON object per line when called with --json
    import subprocess

    proc = subprocess.Popen(
        ["python", "-m", "wikiextractor", str(dump_path), "--json", "-o", "-"],
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
    )
    assert proc.stdout is not None
    for raw in proc.stdout:
        try:
            obj = json.loads(raw)
            if obj.get("text"):
                yield obj
        except json.JSONDecodeError:
            continue


def clean_text(text: str) -> str:
    """Normalise Unicode, collapse whitespace, strip control characters."""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)  # control chars
    text = re.sub(r" {2,}", " ", text)
    return text.strip()


def article_to_record(article: dict) -> dict:
    """Convert a wikiextractor article dict to the hub JSONL schema.

    Schema: {"id": str, "text": str, "meta": {...}}
    """
    text = clean_text(article["text"])
    return {
        "id": article["id"],
        "text": text,
        "meta": {
            "title": article.get("title", ""),
            "url": article.get("url", ""),
            "char_count": len(text),
            "word_count": len(text.split()),
        },
    }


logger.info("Article processing helpers defined")

In [ ]:
from datasketch import MinHash, MinHashLSH

NUM_PERM = 128
JACCARD_THRESHOLD = 0.8


def minhash(text: str, num_perm: int = NUM_PERM) -> MinHash:
    """Compute MinHash signature for near-duplicate detection.

    MinHash LSH deduplication following Broder (1997).
    """
    m = MinHash(num_perm=num_perm)
    for word in text.lower().split():
        m.update(word.encode("utf-8"))
    return m


def build_jsonl_shards(
    dump_path: Path,
    output_dir: Path,
    shard_size: int = SHARD_SIZE,
) -> list[Path]:
    """Stream Wikipedia dump → deduplicated JSONL shards.

    Returns list of written shard paths.
    """
    lsh = MinHashLSH(threshold=JACCARD_THRESHOLD, num_perm=NUM_PERM)
    shard_paths: list[Path] = []
    shard_idx = 0
    records: list[str] = []
    total = 0
    duplicates = 0

    for article in iter_wiki_articles(dump_path):
        record = article_to_record(article)
        mh = minhash(record["text"])
        key = record["id"]

        # Near-duplicate check
        if lsh.query(mh):
            duplicates += 1
            continue
        lsh.insert(key, mh)

        records.append(json.dumps(record, ensure_ascii=False))
        total += 1

        if len(records) >= shard_size:
            path = output_dir / f"shard_{shard_idx:05d}.jsonl"
            path.write_text("\n".join(records), encoding="utf-8")
            logger.info("Wrote %s (%d articles)", path.name, len(records))
            if not WANDB_DISABLED:
                wandb.log({"articles_processed": total, "duplicates_skipped": duplicates})
            shard_paths.append(path)
            records = []
            shard_idx += 1

    # Flush remainder
    if records:
        path = output_dir / f"shard_{shard_idx:05d}.jsonl"
        path.write_text("\n".join(records), encoding="utf-8")
        shard_paths.append(path)
        logger.info("Wrote final shard %s (%d articles)", path.name, len(records))

    logger.info("Total articles: %d | Duplicates skipped: %d", total, duplicates)
    return shard_paths


# --- Run the pipeline (uncomment when dump is available) ---
# shard_paths = build_jsonl_shards(DUMP_PATH, OUTPUT_DIR)
logger.info("Pipeline definition complete — set WIKI_DUMP_PATH and uncomment to run")

In [ ]:
# HuggingFace Dataset export
# Reference: Wolf et al. (2020) https://arxiv.org/abs/1910.03771

from datasets import Dataset, DatasetDict
from huggingface_hub import HfApi


def export_to_hf_hub(
    output_dir: Path,
    repo_id: str,
    private: bool = True,
) -> None:
    """Load sharded JSONL files and push as a HuggingFace Dataset."""
    from datasets import load_dataset

    jsonl_files = sorted(output_dir.glob("shard_*.jsonl"))
    if not jsonl_files:
        logger.error("No JSONL shards found in %s", output_dir)
        return

    ds = load_dataset(
        "json",
        data_files={"train": [str(p) for p in jsonl_files]},
        split="train",
    )
    logger.info("Loaded dataset: %s", ds)

    ds.push_to_hub(repo_id, private=private)
    logger.info("Pushed dataset to HuggingFace Hub: %s", repo_id)

    if not WANDB_DISABLED:
        artifact = wandb.Artifact("wiki-dataset", type="dataset")
        artifact.metadata = {"hf_repo": repo_id, "num_shards": len(jsonl_files)}
        wandb.log_artifact(artifact)


# --- Export (uncomment after build_jsonl_shards has run) ---
# export_to_hf_hub(OUTPUT_DIR, HF_REPO_ID)
logger.info("HF export helper defined")